# Dhaka Bus Network as a Graph — live demo (CSE173)

Run the cells from top to bottom (**Run All**). The first half works on the 11-stop
sample so every number can be checked by hand; the second half runs the same code on
the Dhaka core network.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.data_prep import prepare
from src.graph_build import build_graph, edges_table, handshake_check, route_graph
from src import algorithms as alg
from src import relations_logic as rl
from src import visualize as viz
%matplotlib inline
print("project folder:", ROOT)

## 1. Build the sample graph
Raw notes → cleaning (spellings mapped to stop IDs) → validation rules → graph.

In [ ]:
data = prepare(ROOT / "data" / "sample")
G = build_graph(data)
print(G.n_vertices, "vertices,", G.n_edges, "edges")
edges_table(G)

## 2. Handshaking theorem as a data check
The sum of all degrees must equal 2|E|. If not, the adjacency list has a bug.

In [ ]:
sum_deg, two_e, ok = handshake_check(G.adj)
print(f"sum of degrees = {sum_deg}, 2|E| = {two_e}, holds: {ok}")
pd.DataFrame({"stop": [G.name(s) for s in G.adj], "degree": [len(G.adj[s]) for s in G.adj]})

## 3. Connectivity: quantifiers, BFS components, Warshall
Statement: ∀u ∀v path(u, v). Its negation ∃u ∃v ¬path(u, v) is what the program searches for.

In [ ]:
print("connected:", rl.connectivity_by_quantifiers(G.adj))
print("components:", [sorted(G.name(s) for s in c) for c in alg.connected_components(G.adj)])

order = sorted(G.adj)
M = rl.adjacency_matrix(G.adj, order)
W = rl.warshall(M)
print("closure is reflexive, symmetric, transitive:", rl.is_reflexive(W), rl.is_symmetric(W), rl.is_transitive(W))
pd.DataFrame(M, index=[G.name(s) for s in order], columns=[G.name(s)[:6] for s in order])

## 4. Weak points
A cut vertex is a stop whose removal disconnects the network. Below: delete Farmgate and count again.

In [ ]:
cuts = alg.cut_vertices(G.adj)
print("cut vertices:", [G.name(v) for v in cuts])
print("bridges:", [(G.name(u), G.name(v)) for u, v in alg.bridges(G.adj)])

f = G.stop_id("Farmgate")
without = {u: {v: w for v, w in nb.items() if v != f} for u, nb in G.adj.items() if u != f}
print("components after removing Farmgate:", [sorted(G.name(s) for s in c) for c in alg.connected_components(without)])

## 5. Fewest stops (BFS) vs shortest distance (Dijkstra)

In [ ]:
s, t = G.stop_id("Mirpur 1"), G.stop_id("Farmgate")
hp = alg.fewest_hops_path(G.adj, s, t)
km, dp = alg.shortest_distance_path(G.adj, s, t)
print(f"fewest stops:      {len(hp)-1} hops, {alg.path_length_km(G.adj, hp):.2f} km  {G.names(hp)}")
print(f"shortest distance: {len(dp)-1} hops, {km:.2f} km  {G.names(dp)}")
pd.DataFrame(alg.dijkstra_trace(G.adj, s, G.name))

In [ ]:
viz.plot_paths(G, hp, dp, ROOT / "results" / "sample" / "figures" / "shortest_path.png",
               "Mirpur 1 → Farmgate: fewest stops vs shortest distance")
plt.show()

## 6. Fewest bus changes (route graph)
Vertices are routes; two routes are linked when they share a stop (a set intersection).

In [ ]:
radj = route_graph(G)
print({r: {o: sorted(G.name(x) for x in shared) for o, shared in nb.items()} for r, nb in radj.items()})
print(alg.fewest_bus_changes(radj, G.route_seq, G.stop_id("Technical"), G.stop_id("Sadarghat")))

## 7. Minimum spanning tree: Kruskal (with trace) and Prim

In [ ]:
trace = []
k = alg.kruskal(G.adj, trace=trace)
p = alg.prim(G.adj)
print(f"Kruskal {alg.total_weight(k):.2f} km, Prim {alg.total_weight(p):.2f} km, {len(k)} edges = |V| - 1 = {G.n_vertices - 1}")
display(pd.DataFrame([(G.name(u), G.name(v), round(w, 2), "added" if a else "skipped: cycle") for u, v, w, a in trace],
                     columns=["u", "v", "km", "decision"]))
viz.plot_mst(G, k, ROOT / "results" / "sample" / "figures" / "mst.png", "Sample: minimum spanning tree")
plt.show()

## 8. The same code on the Dhaka core network
`run_analysis` writes everything to `results/dhaka/`. Here we show the key numbers and figures.

In [ ]:
from src.analysis import run_analysis

D_data = prepare(ROOT / "data" / "dhaka", verbose=False)
D = build_graph(D_data)
pairs = pd.read_csv(ROOT / "data" / "dhaka" / "pairs.csv")
res = run_analysis(D, D_data, "dhaka", pairs, ROOT / "results" / "dhaka", echo=False)
plt.close("all")
{k: res[k] for k in ["V", "E", "density", "components", "largest", "cut_vertices", "bridges",
                     "avg_hops", "diameter_hops", "mst_km", "total_km"]}

In [ ]:
for name in ["network", "shortest_path", "cut_points", "mst"]:
    display(Image(filename=str(ROOT / "results" / "dhaka" / "figures" / f"{name}.png"), width=620))

In [ ]:
pd.read_csv(ROOT / "results" / "dhaka" / "shortest_paths.csv")[
    ["from", "to", "fewest_hops", "fewest_hops_km", "shortest_km", "shortest_km_hops", "bus_changes", "routes"]]

## 9. Interactive map
Pan, zoom and hover over links. The layer box (top right) turns the tree on and off. The same map is saved as `results/dhaka/map.html`.

In [ ]:
viz.build_map(D, mst_edges=alg.kruskal(D.adj), cuts=alg.cut_vertices(D.adj))